In [ ]:
import os
import csv
import shutil
from pathlib import Path
import numpy as np
import kagglehub

from pyAudioAnalysis import audioTrainTest as aT
from pyAudioAnalysis import MidTermFeatures as mtf

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.cluster import KMeans
from scipy.stats import mode

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def load_complete_map(dataset_path):
    csv_demographics = list(Path(dataset_path).rglob("speaker_demographics.csv"))[0]

    gender_map = {}
    with open(csv_demographics, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            sid = row["speakerId"].strip()
            g = row["gender"].strip().lower()

            if g.startswith("f"):
                gender_map[sid] = "F"
            elif g.startswith("m"):
                gender_map[sid] = "M"

    audio_metadata = []

    csv_files = [p for p in Path(dataset_path).rglob("*.csv")
                 if "demographics" not in p.name]

    for csv_file in csv_files:
        with open(csv_file, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)

            for row in reader:
                sid = row.get("speakerId", "").strip()
                rel_path = row.get("path", "").strip()
                obj = row.get("object", "unknown").strip().lower()

                gender = gender_map.get(sid)

                if gender and rel_path:
                    audio_metadata.append({
                        "rel_path": rel_path,
                        "speaker_id": sid,
                        "gender": gender,
                        "object": obj
                    })

    print(f"Loaded metadata: {len(audio_metadata)} samples")
    return audio_metadata

In [ ]:
def resolve_audio_path(dataset_path, rel_path):
    dataset_path = Path(dataset_path)

    # Extract the nested top-level directory name dynamically if needed,
    # or explicitly add the known folder to the search tree
    nested_dir = dataset_path / "fluent_speech_commands_dataset"

    candidates = [
        dataset_path / rel_path,
        nested_dir / rel_path,  # <-- Added this critical candidate
        dataset_path / Path(rel_path).name,
        nested_dir / Path(rel_path).name,
        dataset_path / "data" / rel_path,
        dataset_path / "wav" / rel_path,
        dataset_path / "audio" / rel_path,
        Path(rel_path)
    ]

    for c in candidates:
        if c.exists():
            return c

    return None

In [ ]:
def train_with_pyaudioanalysis(audio_metadata, dataset_path,
                               target_object=None,
                               n_per_gender=20,
                               model_type="svm_rbf"):

    test_name = f"pyaudio_{target_object}" if target_object else "pyaudio_global"
    base_dir = Path(f"./{test_name}_workspace")

    if base_dir.exists():
        shutil.rmtree(base_dir)

    dir_f = base_dir / "females"
    dir_m = base_dir / "males"

    dir_f.mkdir(parents=True, exist_ok=True)
    dir_m.mkdir(parents=True, exist_ok=True)

    if target_object:
        filtered = [m for m in audio_metadata if target_object in m["object"]]
    else:
        filtered = audio_metadata

    counters = {"F": 0, "M": 0}
    seen = set()

    for m in sorted(filtered, key=lambda x: x["speaker_id"]):

        sid = m["speaker_id"]
        gender = m["gender"]

        if sid in seen or counters[gender] >= n_per_gender:
            continue

        path = resolve_audio_path(dataset_path, m["rel_path"])
        if path is None:
            continue

        dest = dir_m if gender == "M" else dir_f
        shutil.copy(path, dest / path.name)

        seen.add(sid)
        counters[gender] += 1

    print(f"Copied: {counters}")

    class_folders = [str(dir_f), str(dir_m)]
    model_name = f"model_{test_name}"

    print("\nTraining SVM...")

    aT.extract_features_and_train(
        class_folders,
        1.0, 1.0,   # mt win/step
        1.0, 1.0,   # st win/step
        model_type,
        model_name,
        False
    )

    print("Training done:", model_name)

    return base_dir, model_name

In [ ]:
def eval_svm(workspace, model_name):
    y_true, y_pred = [], []

    for label, cls in enumerate(["females", "males"]):
        folder = Path(workspace) / cls

        for f in folder.glob("*.wav"):
            try:
                pred = aT.file_classification(str(f), model_name, "svm_rbf")
                y_pred.append(int(float(pred[0])))
                y_true.append(label)
            except:
                pass

    return confusion_matrix(y_true, y_pred)

In [ ]:
from pyAudioAnalysis import audioBasicIO

def extract_features(file_path):
    # 1. Read the audio file to obtain sampling rate (fs) and the raw signal matrix
    fs, signal = audioBasicIO.read_audio_file(file_path)

    # 2. If the audio is stereo (2 channels), convert it to mono by selecting the first channel
    if len(signal.shape) > 1:
        signal = signal[:, 0]

    # 3. Set window parameters in seconds
    mid_window_sec = 1.0
    mid_step_sec = 1.0
    short_window_sec = 0.5
    short_step_sec = 0.5

    # 4. Pass all 6 required arguments to mid_feature_extraction (converting seconds to samples)
    F, _, _ = mtf.mid_feature_extraction(
        signal,
        fs,
        int(mid_window_sec * fs),
        int(mid_step_sec * fs),
        int(short_window_sec * fs),
        int(short_step_sec * fs)
    )

    # 5. Return the mean across time frames
    return np.mean(F, axis=1)

In [ ]:
def eval_kmeans(workspace):
    X, y = [], []

    for label, cls in enumerate(["females", "males"]):
        folder = Path(workspace) / cls

        for f in folder.glob("*.wav"):
            try:
                features = extract_features(str(f))
                if features is not None and len(features) > 0:
                    X.append(features)
                    y.append(label)
            except Exception as e:
                # Log the error so you know WHY it failed instead of hiding it
                print(f"Error processing {f.name}: {e}")

    X = np.array(X)
    y = np.array(y)

    # Fallback check to prevent the KMeans ValueError
    if X.size == 0 or X.ndim < 2:
        print(f"Warning: No features extracted in workspace '{workspace}'. Skipping KMeans.")
        # Return an empty confusion matrix or dummy data to let the loop continue
        return np.zeros((2, 2), dtype=int)

    km = KMeans(n_clusters=2, n_init=10, random_state=42)
    clusters = km.fit_predict(X)

    mapping = {}
    for c in [0,1]:
        idx = np.where(clusters == c)[0]
        mapping[c] = mode(y[idx], keepdims=True).mode[0]

    pred = np.array([mapping[c] for c in clusters])

    return confusion_matrix(y, pred)

In [ ]:
def plot_cm(cm, title):
    # Crea automaticamente la cartella plots se non esiste
    plots_dir = Path("./plots")
    plots_dir.mkdir(exist_ok=True)

    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.title(title, weight='bold')
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.xticks([0, 1], ["Female", "Male"])
    plt.yticks([0, 1], ["Female", "Male"])

    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > (cm.max() / 2) else "black",
                     weight="bold", fontsize=12)

    plt.tight_layout()

    # Pulisce il titolo per usarlo come nome di file valido
    clean_filename = title.lower().replace(" ", "_")
    plot_path = plots_dir / f"confusion_matrix_{clean_filename}.png"
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f"Grafico salvato in: {plot_path}")

def plot_summary(svm_acc, km_acc, labels):
    plots_dir = Path("./plots")
    plots_dir.mkdir(exist_ok=True)

    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(8, 5))
    bars_svm = plt.bar(x - width/2, svm_acc, width, label='SVM', color='#34495e')
    bars_km = plt.bar(x + width/2, km_acc, width, label='KMeans', color='#1abc9c')

    plt.ylabel('Accuracy Score')
    plt.title('Model Accuracy Performance Comparison', weight='bold')
    plt.xticks(x, labels)
    plt.ylim(0, 1.1)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.legend()

    # Aggiunge i valori numerici sopra le barre
    for bars in [bars_svm, bars_km]:
        for bar in bars:
            yval = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02,
                     f"{yval:.2%}", ha='center', va='bottom', weight='bold', fontsize=9)

    plt.tight_layout()
    summary_path = plots_dir / "final_accuracy_comparison.png"
    plt.savefig(summary_path, dpi=150)
    plt.show()
    print(f"Grafico riassuntivo finale salvato in: {summary_path}")

In [ ]:
dataset_path = Path(kagglehub.dataset_download(
    "tommyngx/fluent-speech-corpus"
))
print("dataset_path:", dataset_path)

In [ ]:
# Diagnostic: find out where the .wav files actually live on disk,
# since resolve_audio_path's guessed candidates may not match the
# real kagglehub download layout. Run this once and inspect the output
# before re-running the training/eval loop below.

print("Top-level contents of dataset_path:")
for p in sorted(dataset_path.iterdir()):
    print(" ", p.name, "[dir]" if p.is_dir() else "[file]")

print()
print("First 5 .wav files found anywhere under dataset_path:")
count = 0
for wav in dataset_path.rglob("*.wav"):
    print(" ", wav.relative_to(dataset_path))
    count += 1
    if count >= 5:
        break

if count == 0:
    print("  No .wav files found at all under dataset_path!")

In [ ]:
audio_metadata = load_complete_map(dataset_path)

# Map each condition name to the filter string matched against the
# `object` column (read at runtime from the dataset itself, e.g.
# "heat", \"lights\", \"volume\", \"none\"). Global = no filter (all objects).
target_map = {
    "Global": None,
    "Light": "lights",
    "Volume": "volume"
}

svm_acc = []
km_acc = []
svm_errors = []
km_errors = []

for name, target in target_map.items():

    # Train first - this builds the workspace folders and the model.
    # Use the base_dir / model_name it actually returns rather than
    # guessing the paths, so eval can't drift out of sync with training.
    base_dir, model_name = train_with_pyaudioanalysis(
        audio_metadata, dataset_path, target_object=target
    )

    # SVM
    cm_svm = eval_svm(base_dir, model_name)
    plot_cm(cm_svm, f"SVM {name}")

    acc_svm = np.trace(cm_svm) / np.sum(cm_svm)
    svm_acc.append(acc_svm)

    svm_errors.append(cm_svm[0, 1] + cm_svm[1, 0])

    # KMeans
    cm_km = eval_kmeans(base_dir)
    plot_cm(cm_km, f"KMeans {name}")

    acc_km = np.trace(cm_km) / np.sum(cm_km)
    km_acc.append(acc_km)

    km_errors.append(cm_km[0, 1] + cm_km[1, 0])

print("SVM accuracies:", svm_acc)
print("KMeans accuracies:", km_acc)

# --- INVOCA IL SALVATAGGIO DEL GRAFICO CONGIUNTO FINALE ---
plot_summary(svm_acc, km_acc, list(target_map.keys()))